In [1]:
import pickle
import pandas
import os
from Tools.DatasetTools.DatasetOperator import Dataset, DatasetTester

In [2]:
target_case = 'EF_nmhcp'

In [3]:
ModelName = 'Kernel Ridge'

In [4]:
import logging
Logger = logging.getLogger()
Logger.setLevel(logging.INFO)
logging.basicConfig(format='%(message)s')

In [5]:
namefile = ModelName.replace(' ', '')
suffix = f"no_hcp_bcc_fcc_{namefile}" #'CV_restart_folds_inloop''CV_restart_folds_inloop

In [6]:
dataset = 'Fe-Mo'

In [7]:
DS = Dataset('Fe-Mo', target_name=target_case, remove_phases_query='Phase != "bcc" and Phase != "fcc" and Phase !="hcp"')

In [8]:
feature_concat_resul_loc = os.path.join(dataset, 'results', f'concatenation_results_{target_case}_{suffix}.pkl')  
if os.path.exists(feature_concat_resul_loc):
    with open(feature_concat_resul_loc, 'rb') as pkl:
        savedFCresults = pickle.load(pkl)
else:
    savedFCresults = {}

In [9]:
savedFCresults[(ModelName, 'atomic no CNAV')]

[                                     train1     test1  \
 MagpieData mean GSmagmom           0.144313  0.146840   
 Structure                          0.127471  0.138823   
 MagpieData maximum Row             0.125671  0.142002   
 MagpieData minimum AtomicWeight    0.123107  0.141536   
 MagpieData mode NsValence          0.115461  0.139977   
 MagpieData avg_dev Number          0.105383  0.141100   
 Mag                                0.097627  0.146766   
 MagpieData mode Electronegativity  0.093689  0.148566   
 
                                                                               params  \
 MagpieData mean GSmagmom           {'regressor__alpha': 1.0, 'regressor__coef0': ...   
 Structure                          {'regressor__alpha': 1.0, 'regressor__coef0': ...   
 MagpieData maximum Row             {'regressor__alpha': 1.0, 'regressor__coef0': ...   
 MagpieData minimum AtomicWeight    {'regressor__alpha': 1.0, 'regressor__coef0': ...   
 MagpieData mode NsValence     

In [11]:
cleanfcresults = {}
for combi, fcresults in savedFCresults.items():
    ncurves = len(fcresults)
    if ModelName not in combi:
        Logger.info(f'{combi} is not {ModelName}')
        continue
    if ncurves < 1:
        Logger.info(f'{combi} has no curves')
        continue
    Logger.info(f'combi: {combi}, curves: {ncurves}')
    cleanfcresults[combi] = []
    for i, curve in enumerate(fcresults):
        if 'random' in curve.index:
            random_in = curve.index.get_loc('random')
        else:
            random_in = len(curve.index)
        cleancurve = curve.iloc[:,:random_in]
        nfeats = cleancurve.shape[0]
        if nfeats > 0:
            cleanfcresults[combi].append(curve)
#            Logger.info(f'curve {i}, nselected = {curve.shape}, random in: {random_in}, features clean: {nfeats}')

combi: ('Kernel Ridge', 'atomic'), curves: 3
combi: ('Kernel Ridge', 'dataset'), curves: 3
combi: ('Kernel Ridge', 'Canonical ACE'), curves: 12
combi: ('Kernel Ridge', 'Canonical BOP'), curves: 13
combi: ('Kernel Ridge', 'ACE no CNAV'), curves: 10
combi: ('Kernel Ridge', 'ACE'), curves: 12
combi: ('Kernel Ridge', '0.7dProjections 0.5OS BOP'), curves: 12
combi: ('Kernel Ridge', '0.7spProjections 0.5OS BOP'), curves: 13
combi: ('Kernel Ridge', 'atomic no CNAV'), curves: 10
combi: ('Kernel Ridge', 'dataset no CNAV'), curves: 10
combi: ('Kernel Ridge', '0.7dProjections 0.5OS BOP no CNAV'), curves: 10
combi: ('Kernel Ridge', '0.7spProjections 0.5OS BOP no CNAV'), curves: 10
combi: ('Kernel Ridge', 'SOAP_specific_small'), curves: 10
combi: ('Kernel Ridge', 'SOAP_specific_small no CNAV'), curves: 10
combi: ('Kernel Ridge', 'Canonical ACE no CNAV'), curves: 10
combi: ('Kernel Ridge', 'Canonical BOP no CNAV'), curves: 67
combi: ('Kernel Ridge', 'SOAP_canonicalW_small'), curves: 10
combi: ('Kern

test by learning

In [10]:
from Tools.DatasetTools.MLConveniences import filter_features
import joblib
dataset='Fe-Mo'

In [11]:
models = [ModelName]

In [12]:
modelnames=[model.replace(' ','') for model in models]

In [13]:
voting_regressor_files = {modelname: os.path.join(dataset, 'results', f'voting_regressor_{modelname}.pkl')
                         for modelname in modelnames}

In [14]:
voting_regressor_files

{'RandomForest': 'Fe-Mo/results/voting_regressor_RandomForest.pkl'}

In [15]:
optimal_regressors = {}
for model in modelnames:
    optimal_regressors.update(joblib.load(voting_regressor_files[model]))

In [16]:
redbold = "\x1b[31;1m"
reset = "\x1b[0m"

In [17]:
problematic_estimators = {}
for ( model, featurename ), votingregressor in optimal_regressors.items():
    estimators = votingregressor.estimators
    Logger.info(f'model:{model}, featurename: {featurename}, nestimators :{len(estimators)}' )
    X = DS.Features[featurename]
    problematic_estimators[(model, featurename)]=[]
    for iestimator, estimator in estimators:
        learning_curve = estimator.named_steps.feature_selection.kw_args['learning_curve']
        nfeatures = learning_curve.shape
        transformed_X =estimator.named_steps.feature_selection.transform(X)
        if transformed_X.shape[1] < 1:
            Logger.warning(f'{redbold}index{iestimator}, transformed curve length is ZERO (0){reset}')
            problematic_estimators[(model, featurename)].append(iestimator)
            continue
        Logger.info(f'index:{iestimator}, transformer curve length: {nfeatures}, nfeatures stranformed: {transformed_X.shape}')


model:Random Forest, featurename: atomic, nestimators :5
index:0, transformer curve length: (8, 5), nfeatures stranformed: (262, 6)
index:1, transformer curve length: (8, 5), nfeatures stranformed: (262, 2)
index:2, transformer curve length: (5, 5), nfeatures stranformed: (262, 3)
index:3, transformer curve length: (8, 5), nfeatures stranformed: (262, 2)
index:4, transformer curve length: (8, 5), nfeatures stranformed: (262, 2)
model:Random Forest, featurename: dataset, nestimators :5
index:0, transformer curve length: (21, 5), nfeatures stranformed: (262, 17)
index:1, transformer curve length: (21, 5), nfeatures stranformed: (262, 10)
index:2, transformer curve length: (20, 5), nfeatures stranformed: (262, 10)
index:3, transformer curve length: (21, 5), nfeatures stranformed: (262, 8)
index:4, transformer curve length: (21, 5), nfeatures stranformed: (262, 11)
model:Random Forest, featurename: Canonical ACE, nestimators :9
index:0, transformer curve length: (197, 5), nfeatures stranfo

In [19]:
problematic = savedFCresults[(ModelName, 'atomic no CNAV')][4]

In [22]:
optimal_regressors[(ModelName, 'atomic no CNAV')]

VotingRegressor(estimators=[('0',
                             Pipeline(steps=[['feature_selection',
                                              FunctionTransformer(func=<function filter_features at 0x7f8ac894fa60>,
                                                                  kw_args={'learning_curve':                                   train1     test1  \
MagpieData mode NsValence       0.148138  0.148498   
MagpieData minimum Row          0.148116  0.148498   
Structure                       0.125774  0.139549   
Mag                             0.120233  0.140391   
MagpieData maximum GSvolume_pa  0.119420  0.140614   
MagpieData me...
                                       test     train  
MagpieData mean GSvolume_pa        0.142018  0.112481  
MagpieData maximum Column          0.142014  0.112481  
MagpieData maximum CovalentRadius  0.142019  0.112483  
Mag                                0.142007  0.110764  
MagpieData mode NsValence          0.141578  0.110953  
MagpieData mode Row                0.141592  0.110960  
Structure                          0.141327  0.084068  ,
                                                                           'remove_structure': True})],
                                             ('regressor',
                                              RandomForestRegressor(max_depth=10,
                                                                    max_leaf_nodes=20))]))])

In [21]:
problematic_estimator = optimal_regressors[(ModelName, 'atomic no CNAV')].estimators.pop(4)

In [23]:
problematic_estimator[1].named_steps.feature_selection.transform(X)

""
Fe_pv4Mo_sv20.C36-ABBBB.FM
Fe_pv15Mo_sv38.R-AAAABBBBBBB.NM
Fe_pv2Mo_sv11.mu-BBABB.FM
Fe_pv8Mo_sv22.sigma-BBBAB.NM
Fe_pv2Mo_sv11.mu-BBBBA.NM
...
Fe_pv8Mo_sv16.C36-BAABB.NM
Fe_pv30.sigma.FM
Fe_pv6.C15.FM
Mo_sv8.A15.NM


In [27]:
DS.Features['atomic no CNAV'][problematic_estimator[1].named_steps.feature_selection.kw_args['learning_curve'].index]

,MagpieData mean NUnfilled,MagpieData mode Number,MagpieData mode NsValence,MagpieData minimum Row,MagpieData maximum AtomicWeight,Mag,MagpieData avg_dev AtomicWeight,Structure
Fe_pv4Mo_sv20.C36-ABBBB.FM,5.666667,42.0,1.0,4.0,95.960,1,11.143056,3
Fe_pv15Mo_sv38.R-AAAABBBBBBB.NM,5.433962,42.0,1.0,4.0,95.960,0,16.280206,4
Fe_pv2Mo_sv11.mu-BBABB.FM,5.692308,42.0,1.0,4.0,95.960,1,10.444142,9
Fe_pv8Mo_sv22.sigma-BBBAB.NM,5.466667,42.0,1.0,4.0,95.960,0,15.689422,10
Fe_pv2Mo_sv11.mu-BBBBA.NM,5.692308,42.0,1.0,4.0,95.960,0,10.444142,9
...,...,...,...,...,...,...,...,...
Fe_pv8Mo_sv16.C36-BAABB.NM,5.333333,42.0,1.0,4.0,95.960,0,17.828889,3
Fe_pv30.sigma.FM,4.000000,26.0,2.0,4.0,55.845,1,0.000000,10
Fe_pv6.C15.FM,4.000000,26.0,2.0,4.0,55.845,1,0.000000,2
Mo_sv8.A15.NM,6.000000,42.0,1.0,5.0,95.960,0,0.000000,0


In [ ]:
Votin

In [26]:
DS.Features['atomic no CNAV'][problematic.index]

,MagpieData mean GSvolume_pa,Mag,MagpieData maximum GSmagmom,Structure,MagpieData mode Row,MagpieData mode NsValence,MagpieData avg_dev AtomicWeight,MagpieData minimum Column
Fe_pv4Mo_sv20.C36-ABBBB.FM,14.863333,1,2.110663,3,5.0,1.0,11.143056,6.0
Fe_pv15Mo_sv38.R-AAAABBBBBBB.NM,14.286226,0,2.110663,4,5.0,1.0,16.280206,6.0
Fe_pv2Mo_sv11.mu-BBABB.FM,14.926923,1,2.110663,9,5.0,1.0,10.444142,6.0
Fe_pv8Mo_sv22.sigma-BBBAB.NM,14.367333,0,2.110663,10,5.0,1.0,15.689422,6.0
Fe_pv2Mo_sv11.mu-BBBBA.NM,14.926923,0,2.110663,9,5.0,1.0,10.444142,6.0
...,...,...,...,...,...,...,...,...
Fe_pv8Mo_sv16.C36-BAABB.NM,14.036667,0,2.110663,3,5.0,1.0,17.828889,6.0
Fe_pv30.sigma.FM,10.730000,1,2.110663,10,4.0,2.0,0.000000,8.0
Fe_pv6.C15.FM,10.730000,1,2.110663,2,4.0,2.0,0.000000,8.0
Mo_sv8.A15.NM,15.690000,0,0.000000,0,5.0,1.0,0.000000,6.0


In [29]:
problematic_estimators

{('Random Forest', 'atomic'): [],
 ('Random Forest', 'dataset'): [],
 ('Random Forest', 'Canonical ACE'): [],
 ('Random Forest', 'Canonical BOP'): [],
 ('Random Forest', '0.7dProjections 0.5OS BOP'): [],
 ('Random Forest', '0.7spProjections 0.5OS BOP'): [],
 ('Random Forest', 'ACE no CNAV'): [],
 ('Random Forest', 'ACE'): [],
 ('Random Forest', 'atomic no CNAV'): ['4'],
 ('Random Forest', 'dataset no CNAV'): [],
 ('Random Forest', '0.7dProjections 0.5OS BOP no CNAV'): [],
 ('Random Forest', '0.7spProjections 0.5OS BOP no CNAV'): [],
 ('Random Forest', 'SOAP_specific_small'): [],
 ('Random Forest', 'SOAP_specific_small no CNAV'): [],
 ('Random Forest', 'Canonical ACE no CNAV'): [],
 ('Random Forest', 'Canonical BOP no CNAV'): [],
 ('Random Forest', 'SOAP_canonicalW_small'): [],
 ('Random Forest', 'SOAP_canonicalW_small no CNAV'): []}

In [ ]:
cleanFCresults = {}

In [ ]:
for (modelname, featurename), listofcurves in savedFCresults

In [ ]:
for (modelname, featurename), list_problem_index  in problematic_estimators:
    cleanFCresults[(modelname, featurename)] = []
    if len(list_problem_index) < 0:
        continue
    for 